# Кросс-валидация - метод оценки обобщающей способности модели

## 1. Загрузка необходимых библиотек и данных

In [39]:
import re
import pandas as pd
import numpy as np

from IPython.display import display

from datasets import load_dataset
from sklearn.datasets import fetch_20newsgroups

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import f1_score

from sklearn.model_selection import StratifiedKFold, cross_validate

import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\alesh\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

### 1.1 Загрузка датасета emotion

In [2]:
dataset_emotion = load_dataset('emotion', split='train+test')

print(f"emotion size: {len(dataset_emotion)}")

emotion size: 18000


### 1.2 Загрузка датачета 20newsgroups(только указанные  категории)

In [3]:
categories = ['comp.sys.ibm.pc.hardware',
              'comp.sys.mac.hardware',
              'comp.graphics',
              'comp.windows.x']

dataset_news = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

print(f"news train size: {len(dataset_news.data)}")

news train size: 3906


## 2. Предобработка текстов

### 2.1 Удаление шума(для news)

In [5]:
def clean_noise(text):
    text = re.sub(r'^(From|Subject|Organization|Lines|Distribution|NNTP-Posting-Host|Keywords|Summary):.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        if len(line) == 0:
            continue
        non_alnum = sum(1 for c in line if not c.isalnum() and not c.isspace())
        if non_alnum/len(line) > 0.7 and len(line) > 10:
            continue
        cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r"[^\w\s.!?]", '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

### 2.2 Токенизация, стемминг, лемматизация, лемматизация(только сущ и прил), LSA

In [9]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [11]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def tokenize_text(text):
    words = nltk.word_tokenize(text.lower())
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

def stem_text(text):
    words = nltk.word_tokenize(text.lower())
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    return ' '.join(words)

def lemmatize_text(text):
    words = nltk.word_tokenize(text.lower())
    pos_tags = nltk.pos_tag(words)
    lemmatized_words = []
    for word, tag in pos_tags:
        if word not in stop_words:
            wordnet_tag = get_wordnet_pos(tag)
            lemmatized_words.append(lemmatizer.lemmatize(word, wordnet_tag))
    return ' '.join(lemmatized_words)

def lemmatize_noun_adj_text(text):
    words = nltk.word_tokenize(text.lower())
    pos_tags = nltk.pos_tag(words)
    lemmatized_words = []
    for word,tag in pos_tags:
        if word not in stop_words:
            if tag.startswith('N') or tag.startswith('J'):
                wordnet_tag = get_wordnet_pos(tag)
                lemmatized_words.append(lemmatizer.lemmatize(word, wordnet_tag))
    return ' '.join(lemmatized_words)

In [12]:
preprocessors= {
    'tokenization': tokenize_text,
    'stemming': stem_text,
    'lemmatization': lemmatize_text,
    'lemmatization_noun_adj': lemmatize_noun_adj_text,
}

### 2.3 Применение предобработки

In [22]:
datasets = {'emotion': dataset_emotion, 'news': dataset_news}
datasets_texts = {}
datasets_labels = {}

In [23]:
datasets_texts['emotion'] = datasets['emotion']['text']
datasets_texts['news'] = [clean_noise(text) for text in datasets['news'].data]

In [24]:
datasets_labels['emotion'] = datasets['emotion']['label']
datasets_labels['news'] = datasets['news'].target

In [25]:
preprocessed_datasets = {'emotion': {}, 'news': {}}

In [28]:
for prep_name, prep_func in preprocessors.items():
    for dataset_name in datasets.keys():
        print(f"Processing {dataset_name} with {prep_name}")
        preprocessed_datasets[dataset_name][prep_name] = [prep_func(text) for text in datasets_texts[dataset_name]]

Processing emotion with tokenization
Processing news with tokenization
Processing emotion with stemming
Processing news with stemming
Processing emotion with lemmatization
Processing news with lemmatization
Processing emotion with lemmatization_noun_adj
Processing news with lemmatization_noun_adj


## 3. Векторизаторы, классификаторы, метрики, кросс-валидаторы

In [29]:
vectorizers = {
    'binary': CountVectorizer(binary=True),
    'count': CountVectorizer(binary=False),
    'tfidf': TfidfVectorizer()
}

In [34]:
classifiers = {
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),
    'RandomForestClassifier': RandomForestClassifier(random_state=42)
}

In [35]:
scoring = ['f1_micro', 'f1_macro', 'f1_weighted']

In [33]:
skf = StratifiedKFold(n_splits=5,shuffle=True, random_state=42)

In [30]:
results = []

## 4. Цикл экспериментов (по всем датасетам, предобработкам, векторизаторам и классификаторам)

In [43]:
for dataset_name in datasets.keys():
    print(f"\n========== {dataset_name.upper()} ==========")
    y = datasets_labels[dataset_name]
    for prep_name in preprocessors.keys():
        X_train_prep = preprocessed_datasets[dataset_name][prep_name]
        for vec_name, vectorizer in vectorizers.items():
            X_vec = vectorizer.fit_transform(X_train_prep)
            for clf_name, clf in classifiers.items():

                print(f"{prep_name} | {vec_name} | {clf_name}")

                scores = cross_validate(clf,X_vec,y,cv=skf, scoring=scoring)

                row = {
                    'Dataset': dataset_name,
                    'Prep': prep_name,
                    'Vectorizer': vec_name,
                    'Classifier': clf_name
                }

                for metric in scoring:
                    micro_mean = np.mean(scores[f"test_{metric}"])
                    micro_std = np.std(scores[f"test_{metric}"])
                    row[f"{metric}_mean"] = micro_mean
                    row[f"{metric}_std"] = micro_std

                results.append(row)


========== EMOTION ==========
tokenization | binary | DecisionTreeClassifier
tokenization | binary | RandomForestClassifier


KeyboardInterrupt: 

## 5. Таблица результатов

In [ ]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by='f1_micro_mean', ascending=False)

In [ ]:
display(df_results)